####Ex 1→ RANK,DENSE_RANK and ROW_NUMBER - know the difference
The three ranking functions behave differently when values are tied. This difference comes up in every DE interview — know which one to use for which situation.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from pyspark.sql import *
from delta.tables import *

schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("emp_name", StringType(), True),
    StructField("dept", StringType(), True),
    StructField("region", StringType(), True),
    StructField("month", StringType(), True),
    StructField("sales", DoubleType(), True),
    StructField("units_sold", IntegerType(), True)
])

data = [
    (1,  "Alice",   "Electronics", "North", "2024-01", 95000.0,  120),
    (2,  "Bob",     "Electronics", "North", "2024-01", 72000.0,   95),
    (3,  "Charlie", "Electronics", "South", "2024-01", 88000.0,  110),
    (4,  "Diana",   "Clothing",    "North", "2024-01", 65000.0,  200),
    (5,  "Eve",     "Clothing",    "South", "2024-01", 78000.0,  185),
    (6,  "Frank",   "Clothing",    "East",  "2024-01", 102000.0, 240),
    (7,  "Grace",   "Grocery",     "North", "2024-01", 45000.0,  500),
    (8,  "Henry",   "Grocery",     "South", "2024-01", 52000.0,  580),
    (9,  "Ivy",     "Grocery",     "East",  "2024-01", 61000.0,  620),
    (10, "Jack",    "Electronics", "East",  "2024-01", 115000.0, 140),
    (1,  "Alice",   "Electronics", "North", "2024-02", 102000.0, 130),
    (2,  "Bob",     "Electronics", "North", "2024-02", 68000.0,   88),
    (3,  "Charlie", "Electronics", "South", "2024-02", 91000.0,  115),
    (4,  "Diana",   "Clothing",    "North", "2024-02", 70000.0,  210),
    (5,  "Eve",     "Clothing",    "South", "2024-02", 83000.0,  195),
    (6,  "Frank",   "Clothing",    "East",  "2024-02", 98000.0,  230),
    (7,  "Grace",   "Grocery",     "North", "2024-02", 48000.0,  520),
    (8,  "Henry",   "Grocery",     "South", "2024-02", 55000.0,  595),
    (9,  "Ivy",     "Grocery",     "East",  "2024-02", 59000.0,  610),
    (10, "Jack",    "Electronics", "East",  "2024-02", 121000.0, 150),
    (1,  "Alice",   "Electronics", "North", "2024-03", 98000.0,  125),
    (2,  "Bob",     "Electronics", "North", "2024-03", 75000.0,  100),
    (3,  "Charlie", "Electronics", "South", "2024-03", 85000.0,  108),
    (4,  "Diana",   "Clothing",    "North", "2024-03", 73000.0,  215),
    (5,  "Eve",     "Clothing",    "South", "2024-03", 80000.0,  190),
    (6,  "Frank",   "Clothing",    "East",  "2024-03", 110000.0, 255),
    (7,  "Grace",   "Grocery",     "North", "2024-03", 51000.0,  540),
    (8,  "Henry",   "Grocery",     "South", "2024-03", 57000.0,  600),
    (9,  "Ivy",     "Grocery",     "East",  "2024-03", 63000.0,  630),
    (10, "Jack",    "Electronics", "East",  "2024-03", 118000.0, 145),
]

df = spark.createDataFrame(data,schema)
df.createOrReplaceTempView("emp_sales")

df.display()

# Window: rank within each dept by sales for Jan 2024
w = Window.partitionBy("dept").orderBy(col("sales").desc())

df_jan = df.filter(col("month").contains("2024-01"))

df_jan.select(
    "emp_name", "dept", "sales",
    rank().over(w).alias("rank"),
    dense_rank().over(w).alias("dense_rank"),
    row_number().over(w).alias("row_number")
).orderBy("dept","rank").display()

# to see a difference clearly - add a tie
tie_data = [(99, "Tiebreaker","Electronics","North","2024-01",95000.0,100)]
df_tie = spark.createDataFrame(tie_data,schema)
df_with_tie = df_jan.union(df_tie)

w2 = Window.partitionBy("dept").orderBy(col("sales").desc())

df_with_tie.filter(col("dept").contains("Electronics")) \
           .select("emp_name", "sales",
            rank().over(w2).alias("rank"),
            dense_rank().over(w2).alias("dense_rank"),
            row_number().over(w2).alias("row_number")       
).orderBy("rank").display()

emp_id,emp_name,dept,region,month,sales,units_sold
1,Alice,Electronics,North,2024-01,95000.0,120
2,Bob,Electronics,North,2024-01,72000.0,95
3,Charlie,Electronics,South,2024-01,88000.0,110
4,Diana,Clothing,North,2024-01,65000.0,200
5,Eve,Clothing,South,2024-01,78000.0,185
6,Frank,Clothing,East,2024-01,102000.0,240
7,Grace,Grocery,North,2024-01,45000.0,500
8,Henry,Grocery,South,2024-01,52000.0,580
9,Ivy,Grocery,East,2024-01,61000.0,620
10,Jack,Electronics,East,2024-01,115000.0,140


emp_name,dept,sales,rank,dense_rank,row_number
Frank,Clothing,102000.0,1,1,1
Eve,Clothing,78000.0,2,2,2
Diana,Clothing,65000.0,3,3,3
Jack,Electronics,115000.0,1,1,1
Alice,Electronics,95000.0,2,2,2
Charlie,Electronics,88000.0,3,3,3
Bob,Electronics,72000.0,4,4,4
Ivy,Grocery,61000.0,1,1,1
Henry,Grocery,52000.0,2,2,2
Grace,Grocery,45000.0,3,3,3


emp_name,sales,rank,dense_rank,row_number
Jack,115000.0,1,1,1
Alice,95000.0,2,2,2
Tiebreaker,95000.0,2,2,3
Charlie,88000.0,4,3,4
Bob,72000.0,5,4,5


####Ex 2→ Top-N per group - the most common DE interview pattern
Find the top N records per group — top 2 salespeople per department, latest order per customer, most recent record per key. This is asked in every single DE interview.

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

# top 2 sales people per department for jan 2024
w = Window.partitionBy("dept").orderBy(col("sales").desc())

top_2 = df.filter(col("month").contains("2024-01")) \
          .withColumn("row_number",row_number().over(w)) \
          .filter(col("row_number") <= 2) \
          .drop("row_number").orderBy("dept",col("sales").desc())

top_2.display()

# top 1 per dept per month (best performer each month)
w = Window.partitionBy("dept","month").orderBy(col("sales").desc())

best_per_month = df.withColumn("row_number",row_number().over(w)) \
                   .filter(col("row_number") == 1) \
                   .select("month","dept","emp_name", "sales") \
                   .orderBy("month","dept")

best_per_month.display()

# bottom 1 per dept - just flip the sort order
w3 = Window.partitionBy("dept").orderBy(col("sales").asc())

bottom_per_dept = df.filter(col("month").contains("2024-01")) \
                    .withColumn("row_number",row_number().over(w3)) \
                    .filter(col("row_number")==1) \
                    .select("emp_name","dept","sales") \
                    .orderBy("dept")

bottom_per_dept.display()

emp_id,emp_name,dept,region,month,sales,units_sold
6,Frank,Clothing,East,2024-01,102000.0,240
5,Eve,Clothing,South,2024-01,78000.0,185
10,Jack,Electronics,East,2024-01,115000.0,140
1,Alice,Electronics,North,2024-01,95000.0,120
9,Ivy,Grocery,East,2024-01,61000.0,620
8,Henry,Grocery,South,2024-01,52000.0,580


month,dept,emp_name,sales
2024-01,Clothing,Frank,102000.0
2024-01,Electronics,Jack,115000.0
2024-01,Grocery,Ivy,61000.0
2024-02,Clothing,Frank,98000.0
2024-02,Electronics,Jack,121000.0
2024-02,Grocery,Ivy,59000.0
2024-03,Clothing,Frank,110000.0
2024-03,Electronics,Jack,118000.0
2024-03,Grocery,Ivy,63000.0


emp_name,dept,sales
Diana,Clothing,65000.0
Bob,Electronics,72000.0
Grace,Grocery,45000.0


####Ex 3→ Running total and running average with SUM/AVG OVER
Running totals and moving averages are used in finance, sales dashboards, and time-series pipelines constantly. The ROWS BETWEEN clause controls the window frame.

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Running total of sales per employee across months
w_running = Window.partitionBy("emp_id").orderBy("month").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# Running average
w_ravg = Window.partitionBy("dept").orderBy("month").rowsBetween(-1, Window.currentRow)
# 2nd month moving average

df.withColumn("running_sales_total",round(sum("sales").over(w_running),2)) \
  .withColumn("running_avg_sales",round(avg("sales").over(w_ravg),2)) \
  .withColumn("dept_2m_moving_avg",round(avg("sales").over(w_ravg),2)) \
  .select("emp_name","dept","month","sales","running_sales_total","running_avg_sales","dept_2m_moving_avg") \
  .filter(col("emp_name").isin("Alice","Jack")).orderBy("emp_name","month").display()

emp_name,dept,month,sales,running_sales_total,running_avg_sales,dept_2m_moving_avg
Alice,Electronics,2024-01,95000.0,95000.0,95000.0,95000.0
Alice,Electronics,2024-02,102000.0,197000.0,108500.0,108500.0
Alice,Electronics,2024-03,98000.0,295000.0,109500.0,109500.0
Jack,Electronics,2024-01,115000.0,115000.0,101500.0,101500.0
Jack,Electronics,2024-02,121000.0,236000.0,106000.0,106000.0
Jack,Electronics,2024-03,118000.0,354000.0,101500.0,101500.0


####Ex 4→ MIN and MAX over a window - compare to partition extremes
MIN and MAX over a window let you compare each row's value to the best and worst in its group — without any self-join or subquery. Common in performance dashboards.

In [0]:
from pyspark.sql import *
from pyspark.sql.types import *
from pyspark.sql.functions import *

w = Window.partitionBy("dept","month")

df.withColumn("dept_best_sales", round(max("sales").over(w),2)) \
       .withColumn("dept_worst_sales", round(min("sales").over(w),2)) \
       .withColumn("gap_from_best", round(max("sales").over(w) - col("sales"),2)) \
       .withColumn("gap_from_worst", col("sales") - round(min("sales").over(w),2)) \
       .withColumn("pct_of_best", round(col("sales") / max("sales").over(w) * 100, 1)) \
       .select(
           "month",
           "emp_name",
           "dept",
           "sales",
           "dept_best_sales",
           "gap_from_best",
           "dept_worst_sales",
           "gap_from_worst",
           "pct_of_best") \
       .filter(col("month").contains("2024-01")) \
       .orderBy("dept",col("sales").desc()).display()

month,emp_name,dept,sales,dept_best_sales,gap_from_best,dept_worst_sales,gap_from_worst,pct_of_best
2024-01,Frank,Clothing,102000.0,102000.0,0.0,65000.0,37000.0,100.0
2024-01,Eve,Clothing,78000.0,102000.0,24000.0,65000.0,13000.0,76.5
2024-01,Diana,Clothing,65000.0,102000.0,37000.0,65000.0,0.0,63.7
2024-01,Jack,Electronics,115000.0,115000.0,0.0,72000.0,43000.0,100.0
2024-01,Alice,Electronics,95000.0,115000.0,20000.0,72000.0,23000.0,82.6
2024-01,Charlie,Electronics,88000.0,115000.0,27000.0,72000.0,16000.0,76.5
2024-01,Bob,Electronics,72000.0,115000.0,43000.0,72000.0,0.0,62.6
2024-01,Ivy,Grocery,61000.0,61000.0,0.0,45000.0,16000.0,100.0
2024-01,Henry,Grocery,52000.0,61000.0,9000.0,45000.0,7000.0,85.2
2024-01,Grace,Grocery,45000.0,61000.0,16000.0,45000.0,0.0,73.8


####Ex 5→ LAG and LEAD - month-over-month change
LAG gets the value from a previous row, LEAD gets the value from a future row — within a defined window. The classic use case is month-over-month or year-over-year comparisons.

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Window per employee ordered by month
w = Window.partitionBy("emp_id", "emp_name").orderBy("month")

df.withColumn("prev_month_sales", round(lag("sales", 1).over(w), 2)) \
        .withColumn("next_month_sales", round(lead("sales", 1).over(w), 2)) \
        .withColumn("mom_change", round(col("sales") - lag("sales", 1).over(w), 2)) \
        .withColumn("mom_growth_pct", round((col("sales") - lag("sales", 1).over(w)) / lag("sales", 1).over(w) * 100, 1)) \
        .withColumn("mom_trend", when(col("sales") > lag("sales", 1).over(w), "Up") \
        .when(col("sales") < lag("sales", 1).over(w), "Down") \
        .otherwise("Flat")) \
        .select(
            "emp_name",
            "dept",
            "month",
            "sales",
            "prev_month_sales",
            "mom_change",
            "mom_growth_pct",
            "mom_trend") \
        .orderBy("emp_name", "month").display()

emp_name,dept,month,sales,prev_month_sales,mom_change,mom_growth_pct,mom_trend
Alice,Electronics,2024-01,95000.0,null,null,null,Flat
Alice,Electronics,2024-02,102000.0,95000.0,7000.0,7.4,Up
Alice,Electronics,2024-03,98000.0,102000.0,-4000.0,-3.9,Down
Bob,Electronics,2024-01,72000.0,null,null,null,Flat
Bob,Electronics,2024-02,68000.0,72000.0,-4000.0,-5.6,Down
Bob,Electronics,2024-03,75000.0,68000.0,7000.0,10.3,Up
Charlie,Electronics,2024-01,88000.0,null,null,null,Flat
Charlie,Electronics,2024-02,91000.0,88000.0,3000.0,3.4,Up
Charlie,Electronics,2024-03,85000.0,91000.0,-6000.0,-6.6,Down
Diana,Clothing,2024-01,65000.0,null,null,null,Flat


####Ex 6→ Deduplicate with ROW_NUMBER - keep latest record per key
The most practical window function pattern in real DE work. When you get duplicate records for the same key, use ROW_NUMBER to pick exactly one — the most recent, highest value, or any other tie-breaking rule.

In [0]:
from pyspark.sql import *
from pyspark.sql.functions import *
from pyspark.sql.types import *
from datetime import *

# Simulate duplicate records — same emp+month arriving twice (common in kafka/API ingestion)
dup_data = [
    (1, "Alice", "Electronics", "North", "2024-01", 95000.0, 120, "2024-01-31 10:00:00"),
    (1, "Alice", "Electronics", "North", "2024-01", 97000.0, 122, "2024-01-31 14:00:00"), # duplicate,later update
    (2, "Bob",   "Electronics", "North", "2024-01", 72000.0,  95, "2024-01-31 09:00:00"),
    (2, "Bob",   "Electronics", "North", "2024-01", 72000.0,  95, "2024-01-31 09:00:00"), # exact duplicate
    (3, "Charlie","Electronics","South", "2024-01", 88000.0, 110, "2024-01-31 11:00:00"),
]
dup_schema = StructType([
    StructField("emp_id",     IntegerType(), True),
    StructField("emp_name",   StringType(),  True),
    StructField("dept",       StringType(),  True),
    StructField("region",     StringType(),  True),
    StructField("month",      StringType(),  True),
    StructField("sales",      DoubleType(),  True),
    StructField("units_sold", IntegerType(), True),
    StructField("updated_at", StringType(),  True),
])

df_dup = spark.createDataFrame(dup_data,dup_schema)
df_dup.display()

# Deduplicate: keep latest record per emp_id + month
w_dedup = Window.partitionBy("emp_id","month").orderBy(col("updated_at").desc())

df_deduped = df_dup.withColumn("row_number",row_number().over(w_dedup)) \
                   .filter(col("row_number") == 1) \
                   .drop("row_number")

print("After dedup done...")
df_deduped.display()

emp_id,emp_name,dept,region,month,sales,units_sold,updated_at
1,Alice,Electronics,North,2024-01,95000.0,120,2024-01-31 10:00:00
1,Alice,Electronics,North,2024-01,97000.0,122,2024-01-31 14:00:00
2,Bob,Electronics,North,2024-01,72000.0,95,2024-01-31 09:00:00
2,Bob,Electronics,North,2024-01,72000.0,95,2024-01-31 09:00:00
3,Charlie,Electronics,South,2024-01,88000.0,110,2024-01-31 11:00:00


After dedup done...


emp_id,emp_name,dept,region,month,sales,units_sold,updated_at
1,Alice,Electronics,North,2024-01,97000.0,122,2024-01-31 14:00:00
2,Bob,Electronics,North,2024-01,72000.0,95,2024-01-31 09:00:00
3,Charlie,Electronics,South,2024-01,88000.0,110,2024-01-31 11:00:00
